# 第13回　「空気を読む」ことの愚かさ：コンドルセの陪審定理
## ―― 独立な多数は賢い。だが空気を読んだ瞬間、その知恵は消える

統計学Ⅱ　2026後期　／　北星学園大学　／　小野原 彩香

---

### このノートの使い方

今期の到達点。第1回で見た予告編 ―― 「平凡な多数でも、独立に判断すれば、多数決はほぼ確実に正解する」 ―― を、いよいよ完全な形で回収する。そして **その奇跡が“空気を読む”ことでどう崩壊するか** を数値で確かめる。

> ⚠️ **はじめに（これは道徳の話ではない）**
> 
> 「空気を読むことの愚かさ」は、人を挑発するための言葉ではない。**コンドルセの陪審定理**（1785年）という、200年以上前から数学的に証明されている事実の話だ。これから、その定理が成り立つ条件（独立）と、条件が壊れたとき（空気を読む）に何が起きるかを、自分の手で確かめる。

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
from scipy import stats
print("準備OK。次のセルへ。")

---
## 1. コンドルセの陪審定理 ―― 独立な多数は賢い

設定はシンプルだ。

- ある問いに正解（YES/NO）がある。
- 一人ひとりは完璧ではないが、コインより少し賢い ―― 正解率 **55%**。
- 全員が **独立に** 判断し、**多数決** で答えを決める。

**コンドルセの陪審定理**：一人の正解率が 0.5 より高く、各人が独立に判断するなら、**多数決の正解率は、人数を増やすほど 1（確実）に近づく**。

第1回で少しだけ見た。今度は人数を1000人まで増やして確かめよう。

In [ ]:
rng = np.random.default_rng(13)
p, 試行 = 0.55, 20000
τ = stats.norm.ppf(p)        # 後で相関版と同じ仕組みを使うための閾値（marginalにP(正解)=p）

def 多数決正解率(n, ρ):
    雰囲気 = rng.normal(size=(試行, 1))               # 全員が共有する『空気』成分
    個人 = rng.normal(size=(試行, n))                 # 各人の独立な判断成分
    X = np.sqrt(ρ) * 雰囲気 + np.sqrt(1 - ρ) * 個人   # ρ=0なら完全独立、ρが大きいほど空気に従う
    正解した = (X < τ)                                # 各人が正解か（周辺確率は常に p）
    return (正解した.sum(axis=1) > n / 2).mean()      # 多数決が正解した割合

人数リスト = [1, 11, 51, 201, 1001]
print("【独立に判断する場合（ρ=0）】一人の正解率は 55% で固定")
独立 = []
for n in 人数リスト:
    a = 多数決正解率(n, ρ=0.0)
    独立.append(a)
    print(f"  {n:>5} 人で多数決 → 正解率 {a:.1%}")

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(人数リスト, 独立, "o-", color="#3949ab")
plt.axhline(p, ls="--", color="gray", label=f"一人の正解率 {p:.0%}")
plt.xscale("log"); plt.ylim(0.4, 1.02)
plt.xlabel("投票する人数（対数）"); plt.ylabel("多数決の正解率")
plt.title("コンドルセの陪審定理：独立なら、人数とともに正解率は1へ")
plt.legend(); plt.show()
print("55%の凡人でも、独立に大勢集めれば、多数決はほぼ確実に正解する。これが『集合知』。")

**たった55%の凡人でも、独立に1000人集めれば多数決は99.9%正解する。** これが「群衆の智慧」の数学的な正体、コンドルセの陪審定理だ。

ここで効いているのは、第6回で学んだことの裏返しだ ―― 独立な判断は、各人の誤りが互いに打ち消し合い、わずかな“正解寄りの偏り”だけが積み重なって増幅される。**独立であることが、集団を賢くしている。**

---
## 2. 「空気を読む」とは、投票を相関させること

では、人々が **空気を読み始めたら** どうなるか。

空気を読むとは、自分だけの判断ではなく、**その場の雰囲気・多数派の気配に自分の答えを寄せる**こと。モデルでは、全員が共有する「雰囲気」成分の重み $\rho$ を上げることで表せる。

- $\rho=0$：誰も空気を読まない＝完全に独立
- $\rho$ が大きいほど：みんなが同じ「空気」に従う＝判断が互いに相関する

一人ひとりの正解率は **55%のまま変えない**。変えるのは「独立か、空気に従うか」だけ。201人の多数決で、$\rho$ を上げていくと正解率はどうなるか？

In [ ]:
n = 201
ρリスト = [0.0, 0.1, 0.2, 0.4, 0.6, 0.8]
正解率たち = [多数決正解率(n, ρ) for ρ in ρリスト]

for ρ, a in zip(ρリスト, 正解率たち):
    読み = "完全に独立" if ρ == 0 else f"空気を読む度 {ρ}"
    print(f"  ρ={ρ}（{読み}）　→ 201人の多数決の正解率 {a:.1%}")

plt.figure(figsize=(7, 4))
plt.plot(ρリスト, 正解率たち, "s-", color="#e8503a")
plt.axhline(p, ls="--", color="gray", label=f"一人の正解率 {p:.0%}")
plt.ylim(0.4, 1.02)
plt.xlabel("空気を読む度合い ρ（0=独立, 大きいほど同調）")
plt.ylabel("201人の多数決の正解率")
plt.title("わずかに空気を読むだけで、集合知は崩れ落ちる")
plt.legend(); plt.show()

**結果は衝撃的だ。** 完全に独立なら201人で約92%。ところが、ほんの少し空気を読むだけ（$\rho=0.2$）で、正解率は **約61%** まで崩れ落ちる ―― ほとんど一人で答えた55%に逆戻りだ。$\rho$ をさらに上げれば、1000人いようと正解率は55%付近に張り付く。

**201人という人数の価値が、ほぼ完全に消えてしまう。** 空気を読むと、みんなが同じ「雰囲気」という1つの判断に従うので、実質的には **たった1人で決めているのと同じ** になるからだ。

---
## 3. 人数を増やしても、空気を読む集団は救われない

「人数を増やせば挽回できるのでは？」――独立ならできる。だが空気を読む（相関した）集団では、**いくら人を増やしても正解率は頭打ち**になる。第6回（相関した標本は $n$ を増やしても下げ止まる）とまったく同じ構造だ。

In [ ]:
人数リスト2 = [1, 11, 51, 201, 1001, 5001]
plt.figure(figsize=(7, 4.5))
for ρ, col in [(0.0, "#3949ab"), (0.2, "#f0a030"), (0.5, "#e8503a")]:
    ys = [多数決正解率(n, ρ) for n in 人数リスト2]
    ラベル = "独立 ρ=0（集合知）" if ρ == 0 else f"空気を読む ρ={ρ}"
    plt.plot(人数リスト2, ys, "o-", color=col, label=ラベル)
plt.axhline(p, ls="--", color="gray", label="一人の正解率 55%")
plt.xscale("log"); plt.ylim(0.4, 1.02)
plt.xlabel("人数（対数）"); plt.ylabel("多数決の正解率")
plt.title("独立なら1へ。空気を読むと、人数を増やしても頭打ち")
plt.legend(); plt.show()
print("独立(青)だけが人数とともに1へ。空気を読む集団(橙・赤)は、何千人いても賢くならない。")

独立な集団（青）だけが、人数とともに正解率を1へ伸ばす。空気を読む集団（橙・赤）は、5000人集めても賢くならない。

> 💬 **今期の結論 ―― これは事実である**
> 
> 集団が賢くなる条件は「人数が多いこと」ではない。**一人ひとりが独立に判断すること**だ。空気を読む・忖度する・多数派に合わせるという行為は、その独立性を自分から手放すこと ―― いわば **集合知の自殺** である。これは精神論でも価値観でもなく、コンドルセの陪審定理が示す数学的な帰結だ。多数が同じ意見でも、それが互いに空気を読んだ結果なら、その一致にはほとんど情報がない。

---
## 4. 集団浅慮（グループシンク）と、独立を守る仕組み

独立が失われた集団が、自信満々に間違った結論へ進む現象を、社会心理学では **集団浅慮（groupthink）** と呼ぶ。歴史上の組織的失敗の多くがこれで説明される。

だからこそ、賢い組織は **独立性を制度で守る**：

- 偉い人から先に意見を言わない（さもないと全員がその「空気」に従う＝$\rho$ が上がる）
- まず各自が **無記名で同時に** 書いて出す（投票・ブレインライティング）
- わざと反対役（悪魔の代弁者）を置く
- 多様な背景の人を入れる（同じ空気を共有させない）

どれも狙いは一つ ―― **判断の独立を守り、$\rho$ を下げること**だ。

---
## 5. ディベート（教室で）

テーマ：**「会議では空気を読むべきか」**。コンドルセとカスケード（第12回）の言葉を使って、賛成・反対に分かれて論じよう。道徳ではなく **数理** で。

- 論点例：空気を読むことに利点はあるか（スピード・摩擦の回避）？ そのとき集団の判断精度はどうなるか？
- 「みんなが賛成しているから安心だ」と言える条件は？ 言えない条件は？
- あなたが意思決定者なら、$\rho$ を下げるためにどんなルールを置くか？

---
## 今日のまとめ

| ポイント | 中身 |
|---|---|
| コンドルセの陪審定理 | 一人の正解率>0.5 かつ **独立** なら、多数決の正解率は人数とともに1へ |
| 空気を読む | 判断が共有の「雰囲気」に相関する（ρが上がる） |
| 崩壊 | ρ=0.2 でも201人の正解率は92%→61%。人数を増やしても頭打ち |
| 集団浅慮 | 独立を失った集団が自信満々に誤る。独立を制度で守るしかない |

- 集団を賢くするのは人数ではなく、**一人ひとりの独立**。
- 「空気を読む」は独立性の自発的放棄＝集合知の自殺。これは **事実** だ。

> **課題（Moodle）**：コンドルセのシミュレーション結果（自動採点）＋「“空気を読む”ことが集団の判断精度を下げる仕組みを、独立性とコンドルセの定理を使って数理的に説明せよ」の記述（到達目標の総仕上げ）。詳しくはMoodleの第13回課題を見ること。

> **次回予告**：第14回「総括：孤高の意思決定者として」。Ⅰ＝直感の敗北、Ⅱ＝集団の敗北。では我々はどう判断すればよいのか。14回の地図を俯瞰し、第1回の自分と再会する。